# Register a model in the MLFlow server

We will use the MlflowClient instance to:

- Register a new version for the experiment green-taxi-trip-duration
- Retrieve the latests versions of the model 
- Transition to "production" and adding annotations to it.

The environment variables are set in the a `.env` file. Create one and add yours.

In [1]:
import mlflow
from mlflow import MlflowClient
import os 
from dotenv import load_dotenv
from datetime import datetime

In [2]:
load_dotenv()
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI")
RUN_ID = os.environ.get("RUN_ID")

Connect to the MLFlow server:

In [3]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

Register the model:

In [4]:
model_uri = f"runs:/{RUN_ID}/model"
client = MlflowClient()
# Create registered model
tags = {"LinearRegression.framework": "Scikit-Learn"}
desc = "Model to predict ride duration in NYC"

client.create_registered_model(name='lr-ride-duration',tags=tags,description=desc)

<RegisteredModel: aliases={}, creation_timestamp=1761316225414, deployment_job_id='', deployment_job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', description='Model to predict ride duration in NYC', last_updated_timestamp=1761316225414, latest_versions=[], name='lr-ride-duration', tags={'LinearRegression.framework': 'Scikit-Learn'}>

If you now browse to the MLflow server UI you will see the new model registered.
[]

Create a model version

In [ ]:
model_version = client.create_model_version(
    name="lr-ride-duration",
    source=model_uri,
    run_id=RUN_ID
)

print(f"Model registered: {model_version.name} v{model_version.version}")


You now also see the model in the MLFlow UI under `Models`.

In [ ]:
model_name = "lr-ride-duration"
# List all model versions
info = client.get_registered_model(name=model_name)
for v in info.latest_versions:
    print(f"version: {v.version}, Aliases: {v.aliases}, description: {v.description}")

We see we only have a version 1 of the model. If we would have registered more artifacts as models under the same name, we would have seen more versions here.
But we want to change the alias to `production` and add a description to it. 

First change the alias of the version to `production`:

In [ ]:
model_version = 1
new_alias = "production"
client.set_registered_model_alias(
    name=model_name,
    version=model_version,
    alias=new_alias
)

And now add the description:

In [ ]:
date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_alias} on {date}"
)

If we look at the model again the aliases change to production and the description is added:

In [ ]:
client.get_model_version(name=model_name, version=model_version).aliases, client.get_model_version(name=model_name, version=model_version).description

In [ ]:
model = client.get_model_version_by_alias("lr-ride-duration", "production")
print(model)

In [ ]:
for v in client.search_model_versions(f"name='{model_name}'"):
    print(v.version, v.aliases)

In [ ]:
model.aliases